# Evaluation on LLaMA 3 + Instruction-Tuned using FLORES+ benchmark for translation tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.2 MB/s eta 0:00:00


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [3]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.0 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
ds_mkd = load_dataset("openlanguagedata/flores_plus", "mkd_Cyrl")

README.md:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/535k [00:00<?, ?B/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/559k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [8]:
ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [9]:
ds_mkd

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [10]:
ds_eng

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [11]:
dataset_mkd = ds_mkd['dev'].to_pandas()
dataset_mkd

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,mkd,Cyrl,mace1250,,Во понеделникот научниците од медицинскиот фак...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,mkd,Cyrl,mace1250,,Водечките истражувачи сметаат дека ова може да...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,mkd,Cyrl,mace1250,,JAS 39C Грипен се сруши на писта околу 9:30 am...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,mkd,Cyrl,mace1250,,"Утврдено е дека пилот бил Дилокрит Патави, вод...",https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,mkd,Cyrl,mace1250,,Локалните медиуми известуваат за аеродромско п...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,mkd,Cyrl,mace1250,,Туристичката сезона на ридските станици во гла...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,mkd,Cyrl,mace1250,,"Меѓутоа, имаат поинаква убавина и шарм во зима...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,mkd,Cyrl,mace1250,,Само неколку авиокомпании сѐ уште нудат тарифи...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,mkd,Cyrl,mace1250,,Меѓу авиокомпаниите што го нудат тоа се „Ер Ка...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [12]:
dataset_eng = ds_eng['dev'].to_pandas()
dataset_eng

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"On Monday, scientists from the Stanford Univer...",https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,eng,Latn,stan1293,,Lead researchers say this may bring early dete...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,eng,Latn,stan1293,,The JAS 39C Gripen crashed onto a runway at ar...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,eng,Latn,stan1293,,The pilot was identified as Squadron Leader Di...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,eng,Latn,stan1293,,Local media reports an airport fire vehicle ro...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,eng,Latn,stan1293,,The tourist season for the hill stations gener...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,eng,Latn,stan1293,,"However, they have a different kind of beauty ...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,eng,Latn,stan1293,,Only a few airlines still offer bereavement fa...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,eng,Latn,stan1293,,"Airlines that offer these include Air Canada, ...",https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [13]:
text_mkd = dataset_mkd['text'].values.tolist()[:100]
text_eng = dataset_eng['text'].values.tolist()[:100]

In [15]:
len(text_mkd)

100

In [14]:
len(text_eng)

100

In [16]:
text_mkd[:10]

['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.',
 'Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.',
 'JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.',
 'Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.',
 'Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.',
 '28-годишниот Вида

In [17]:
text_eng[:10]

['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.',
 'Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'The pilot was identified as Squadron Leader Dilokrit Pattavee.',
 'Local media reports an airport fire vehicle rolled over while responding.',
 '28-year-old Vidal had joined Barça three seasons ago, from Sevilla.',
 'Since moving to the Catalan-capital, Vidal had played 49 games for the club.',
 "The protest started around 11:00 local time (UTC+1) on 

In [18]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

In [19]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [21]:
def get_prompt_mkd(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"


In [22]:
def get_prompt_srb(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"

In [23]:
def get_prompt_bug(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОтговор ({language}):"

In [24]:
def get_prompt_eng(instruction,text,language):
  return f"{instruction}\nText:{text}\nResponse ({language}):"

In [25]:
def batch_translate(texts,instruction, batch_size=8, max_new_tokens=256,lang_code="eng",language="English"):
    all_preds = []
    tokenizer.pad_token = tokenizer.eos_token
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        if lang_code=="mkd":
          prompts = [get_prompt_mkd(instruction,t,language) for t in batch]
        elif lang_code == "bug":
          prompts = [get_prompt_bug(instruction,t,language) for t in batch]
        elif lang_code == "srb":
          prompts = [get_prompt_srb(instruction,t,language) for t in batch]
        else:
          prompts = [get_prompt_eng(instruction,t,language) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for prompt, out in zip(prompts, decoded):
            translation = out.replace(prompt, "").strip()
            print(f"Batch {i}")
            print(f"{translation} \n")
            all_preds.append(translation)

    return all_preds



In [26]:
from nltk.tokenize import sent_tokenize

def first_sentence(text):
    sents = sent_tokenize(text)
    return sents[0] if sents else text

In [27]:
bleu = load('bleu')

## Few-Shot Mono-lingual prompt (MKD->ENG)

In [28]:
instruction = f"""Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Одговор (Англиски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  Македонски: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Одговор (Англиски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Пример 3
  Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Одговор (Англиски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Пример 4
  Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Одговор (Англиски): This study focuses on the development of new methods for data analysis.
  Пример 5
  Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Одговор (Англиски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [29]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="mkd",language="Англиски")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.Преведување на текстот од македонски на англиски:
  Пример 1
  English: At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  English: A sports match lasts 90 minutes, 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries. 
Преведување на текстот од македонски на англиски:
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV a

In [30]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

["On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.Преведување на текстот од македонски на англиски:\n  Пример 1\n  English: At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.",
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi,

In [31]:
predictions_new  = [text.split("Преведување")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски.',
 'Local media report on an airport fire truck that overturned while firefighters in it responde

In [32]:
predictions_new  = [text.split("Преведи")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 "28-year-old Vidal from Seville joined Barcelona's team.",
 'After moving to the c

In [33]:
predictions_new  = [text.split("Преводите")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 "28-year-old Vidal from Seville joined Barcelona's team.",
 'After moving to the c

In [34]:
predictions_new  = [text.split("\n  Пример")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs about one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 "28-year-old Vidal from Seville joined Barcelona's team.",
 'After moving to the c

In [35]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [36]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3484608560676359,
 'precisions': [0.6582181259600615,
  0.4069488817891374,
  0.281198003327787,
  0.1957465277777778],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0399361022364217,
 'translation_length': 2604,
 'reference_length': 2504}

In [37]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 53.59


## Few-Shot Cross-lingual prompt (MKD->ENG) in english language

In [38]:
instruction = f"""Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Example 2
  Macedonian: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Response (English): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Example 3
  Macedonian: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Response (English): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Example 4
  Macedonian: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Response (English): This study focuses on the development of new methods for data analysis.
  Example 5
  Macedonian: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Response (English): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [39]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128)

Batch 0
On Monday, scientists from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.  Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of 

Batch 0
The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with a low standard of living, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries. 
Example 2
Macedonian: Во 2011 година, во САД се случиле 1,9 милиони случаи на рак и 571,000 смрти од рак.
Response (English): In 2011, there were 1.9 million cases of cancer and 571,000 dea

In [48]:
predictions = [elem.split("Example")[0] for elem in predictions_eng]
predictions

["On Monday, scientists from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.  Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.\nResponse (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of",
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with a low standard of living, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries. \n',
 "The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights. \nText:Во 11 часот е договорена средба по

In [49]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['On Monday, scientists from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with a low standard of living, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters inside it responded to a call.',
 'The 28-year-old Vidal from Seville joined the Barcelona team.',
 'A

In [50]:
predictions_new = [text.split("\nПрвата")[0] for text in predictions_new]
predictions_new

['On Monday, scientists from the medical school at Stanford University announced that they had invented a new tool for diagnosis that sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with a low standard of living, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters inside it responded to a call.',
 'The 28-year-old Vidal from Seville joined the Barcelona team.',
 'A

In [51]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [52]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.33463191050495666,
 'precisions': [0.6386175807663411,
  0.397736143637783,
  0.26929325751421607,
  0.18331922099915327],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0630990415335464,
 'translation_length': 2662,
 'reference_length': 2504}

In [53]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 56.15


## Few-Shot Cross-lingual prompt (MKD->ENG) in serbian language

In [54]:
instruction= f"""Преведи следећи текст са **Македонског** на **Енглески** језик.
Само испиши превод. Не додавај никаква објашњења.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Одговор (Енглески): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Одговор (Енглески): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Одговор (Енглески): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Одговор (Енглески): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""


In [55]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="srb",language="Енглески")

Batch 0
On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs around one American cent. 

Пример 6
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, may be twice as low as in wealthier countries. 

Пример 6
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o 

Batch 0
The JAS 39C Gripen crashed on the runway around 9

In [56]:
predictions = [elem.split("Пример")[0] for elem in predictions_eng]
predictions

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs around one American cent. \n\n',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, may be twice as low as in wealthier countries. \n\n',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights. Превод: The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights. \n\n',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron. \n\n',
 'Loc

In [57]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving

In [58]:
predictions_new  = [text.split("Превод")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving

In [59]:
predictions_new  = [text.split("Протестантите")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, and that costs around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving

In [60]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [61]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3530523118338119,
 'precisions': [0.6544145509662751,
  0.41197321780228435,
  0.2861828618286183,
  0.2013681060282172],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0539137380191694,
 'translation_length': 2639,
 'reference_length': 2504}

In [62]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 51.88


## Few-Shot Cross-lingual prompt (MKD->ENG) in bulgarian language

In [63]:
instruction = f"""Преведи следния текст от **Македонски** на **Английски** език.
Само отпечатай превода на английски. Не добавяй никакви обяснения.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Отговор (Английски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Отговор (Английски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Отговор (Английски): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Отговор (Английски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [64]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="bug",language="Английски")

Batch 0
On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.Пример 6
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.Пример 6
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, 

Batch 0
The JAS 39C Gripen crashed on the runway arou

In [65]:
predictions = [elem.split("Пример")[0] for elem in predictions_eng]
predictions

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights. Преведено на английски: The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights. \n\n',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Loca

In [66]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving t

In [67]:
predictions_new  = [text.split("Преведено")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving t

In [68]:
predictions_new  = [text.split("Преведи")[0] for text in predictions_new]
predictions_new

['On Monday, researchers from the medical school at Stanford University announced that they had invented a new tool for diagnosis, which sorts cells by type: a small chip for printing that can be produced with the help of standard inkjet printers, at a cost of around one American cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from breast cancer, for example, can be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters in it responded to a call.',
 '28-year-old Vidal from Seville joined the Barcelona team.',
 'After moving t

In [69]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [70]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3519626259554145,
 'precisions': [0.6527514231499051,
  0.41183431952662725,
  0.28542094455852157,
  0.2],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0523162939297124,
 'translation_length': 2635,
 'reference_length': 2504}

In [71]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 51.94


## Few-Shot Mono-lingual prompt (ENG->MKD)

In [74]:
instruction = f"""Translate the following text from **English** into **Macedonian**. Only output the translation. Do not add explanations.
  Example 1
  English: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Response (Macedonian): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Example 2
  English: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Response (Macedonian): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Example 3
  English: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Response (Macedonian): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Example 4
  English: This study focuses on the development of new methods for data analysis.
  Response (Macedonian)): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Example 5
  English: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Response (Macedonian): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [75]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,language="Macedonian")

Batch 0
Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на нов дијагностички алатка која може да подредува клетки по тип: малиот штампан чип кој може да се произведува со стандардни штампачи за туш за можно околу еден цент американски долар по примерок. 
The tool, called the "lab-on-a-chip," is a tiny chip that can be manufactured using standard ink 

Batch 0
Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациенти во земји со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.  Example 1
  English: On Monday, at 11 o'clock, a meeting has 

Batch 0
JAS 39C Gripen се срушил врз една писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.  The pilot, a Swedish Air Force officer, was killed in the crash.  The cause of the crash is 

In [76]:
predictions = [elem.split("Example")[0] for elem in predictions_mkd]
predictions

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на нов дијагностички алатка која може да подредува клетки по тип: малиот штампан чип кој може да се произведува со стандардни штампачи за туш за можно околу еден цент американски долар по примерок. \nThe tool, called the "lab-on-a-chip," is a tiny chip that can be manufactured using standard ink',
 'Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациенти во земји со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.  ',
 'JAS 39C Gripen се срушил врз една писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.  The pilot, a Swedish Air Force officer, was killed in the crash.  The cause of the crash is still unknown.  The airport was closed to commercial flights until further no

In [77]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на нов дијагностички алатка која може да подредува клетки по тип: малиот штампан чип кој може да се произведува со стандардни штампачи за туш за можно околу еден цент американски долар по примерок.',
 'Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациенти во земји со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил врз една писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од војсководите Дилокрит Патавеј.',
 'Локалните медиуми извештуваат за пожарна возила кои се срушиле додека одговарале.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се преселил во каталонска

In [78]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [79]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.17879495999574196,
 'precisions': [0.5122850122850123,
  0.23398804440649018,
  0.12979482604817127,
  0.07699486700886607],
 'brevity_penalty': 0.9610570181004163,
 'length_ratio': 0.9617959826703426,
 'translation_length': 2442,
 'reference_length': 2539}

In [80]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 53.72


## Few-Shot Cross-lingual prompt (ENG->MKD) in macedonian language

In [81]:
instruction = f"""Преведи го следниот текст од **Англиски** во **Македонски**. Само испечати го преводот на македонски јазик. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Англиски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Одговор (Македонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Пример 2
  Англиски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Одговор (Македонски): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Пример 3
  Англиски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Одговор (Македонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Пример 4
  Англиски: This study focuses on the development of new methods for data analysis.
  Одговор (Македонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Пример 5
  Англиски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Одговор (Македонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [82]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="mkd",language="Македонски")

Batch 0
Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на ново дијагностичко орудие што може да подредува клетки по тип: мал printable чип што може да се произведува со стандардни инкџет принтери за можно една U.S. цент за секоја.  Преведување на текстот на македонски јазик:Во понеделник, научни 

Batch 0
Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како ракот на дојката може да биде половина од оној во земјите со поголем приход.  Преведување на текстот од англиски на македонски ј 

Batch 0
JAS 39C Gripen се срушил врз пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.  Преведување на текстот од **Англиски** во **Македонски**.  Преведување на текстот од **Англиски** во **Македонски**. 

Batch 0
Пилотот бил идентификуван како еде

In [83]:
predictions  = [text.split("Пример")[0] for text in predictions_mkd]
predictions

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на ново дијагностичко орудие што може да подредува клетки по тип: мал printable чип што може да се произведува со стандардни инкџет принтери за можно една U.S. цент за секоја.  Преведување на текстот на македонски јазик:Во понеделник, научни',
 'Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како ракот на дојката може да биде половина од оној во земјите со поголем приход.  Преведување на текстот од англиски на македонски ј',
 'JAS 39C Gripen се срушил врз пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.  Преведување на текстот од **Англиски** во **Македонски**.  Преведување на текстот од **Англиски** во **Македонски**.',
 'Пилотот бил идентификуван како еден од пилотите на воздухо

In [84]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле откривање на ново дијагностичко орудие што може да подредува клетки по тип: мал printable чип што може да се произведува со стандардни инкџет принтери за можно една U.S. цент за секоја.',
 'Лидните истражувачи велат дека со тоа можат да се појават раниот детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил врз пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд.',
 'Локалните медиуми извештавале за пожарна возило на аеродромот кое се преокрлило додека се одвивала одговорност.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се пресе

In [85]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [86]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.1791134023579089,
 'precisions': [0.5142045454545454,
  0.23477157360406092,
  0.13030035335689047,
  0.07390300230946882],
 'brevity_penalty': 0.970020269134274,
 'length_ratio': 0.9704608113430484,
 'translation_length': 2464,
 'reference_length': 2539}

In [87]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 48.55


## Few-Shot Cross-lingual prompt (ENG->MKD) in serbian language

In [88]:
instruction = f"""Преведи следећи текст са **Eнглеског** на **Mакедонски** језик.
Само испиши превод на македонски језик. Не додавај никаква објашњења или додатне примере.

Пример 1
Енглески: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Одговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Енглески: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Одговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Енглески: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Одговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Енглески: This study focuses on the development of new methods for data analysis.
Одговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Енглески: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Одговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [89]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="srb",language="Македонски")

Batch 0
Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: мали штампани чипови кои можат да се произведуваат со стандардни штампачи за туш за можно една американска цента по примерок.

Пример 7
Енглески: The new tool is a tiny printable chip that can be manufactured using standard 

Batch 0
Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход. 

Пример 6
Енглески: The new method is based on the principle of the "differential 

Batch 0
JAS 39C Gripen се срушил на писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови. 

Пример 6
Енглески: The new system will be implemented in the next quarter.
Одговор (Македонски):

In [90]:
predictions  = [text.split("Пример")[0] for text in predictions_mkd]
predictions

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: мали штампани чипови кои можат да се произведуваат со стандардни штампачи за туш за можно една американска цента по примерок.\n\n',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход. \n\n',
 'JAS 39C Gripen се срушил на писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови. \n\n',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван к

In [91]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: мали штампани чипови кои можат да се произведуваат со стандардни штампачи за туш за можно една американска цента по примерок.',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил на писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд.',
 'Локалните медиуми извештавале за пожарна кола која се преокренала додека одговарала на повик.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се преселил во каталонска

In [93]:
predictions_new = [text.split("\n\n")[0] for text in predictions]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: мали штампани чипови кои можат да се произведуваат со стандардни штампачи за туш за можно една американска цента по примерок.',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход. ',
 'JAS 39C Gripen се срушил на писта околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови. ',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд. Пилотот бил идентификуван како еден од 

In [94]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [95]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.18509415143742372,
 'precisions': [0.5098580441640379,
  0.23850574712643677,
  0.13142123287671234,
  0.07379248658318426],
 'brevity_penalty': 0.9988177341279387,
 'length_ratio': 0.9988184324537219,
 'translation_length': 2536,
 'reference_length': 2539}

In [96]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 50.46


## Few-Shot Cross-lingual prompt (ENG->MKD) in bulgarian language

In [97]:
instruction = f"""Преведи следния текст от **Aнглийски** на **Mакедонски** език.
Само отпечатай превода на македонски език. Не добавяй никакви обяснения или допълнителни примери.

Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Отговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Английски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Отговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Английски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Отговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Английски: This study focuses on the development of new methods for data analysis.
Отговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Английски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Отговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""


In [98]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="bug",language="Македонски")

Batch 0
Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: малиот штампан чип што може да се произведува со стандардни штампачи за туш за можно една американска цента.

Пример 6
Английски: The new diagnostic tool is a tiny printable chip that can be manufactured using standard inkjet printers for possibly 

Batch 0
Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.  Текст:This is the first time that the technology has been used in a real-world setting, and the researchers 

Batch 0
JAS 39C Gripen се срушил на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.  Срушил се на пистата околу 9:30 утрински часо

In [99]:
predictions  = [text.split("Пример")[0] for text in predictions_mkd]
predictions

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: малиот штампан чип што може да се произведува со стандардни штампачи за туш за можно една американска цента.\n\n',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.  Текст:This is the first time that the technology has been used in a real-world setting, and the researchers',
 'JAS 39C Gripen се срушил на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.  Срушил се на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијал',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната

In [100]:
predictions_new = [first_sentence(text) for text in predictions]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: малиот штампан чип што може да се произведува со стандардни штампачи за туш за можно една американска цента.',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд.',
 'Локалните медиуми извештавале за пожарна кола која се преокренала додека одговарала на повик.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се преселил во каталонската престолнина,

In [101]:
predictions_new = [text.split("\n\n")[0] for text in predictions_new]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: малиот штампан чип што може да се произведува со стандардни штампачи за туш за можно една американска цента.',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд.',
 'Локалните медиуми извештавале за пожарна кола која се преокренала додека одговарала на повик.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се преселил во каталонската престолнина,

In [102]:
predictions_new = [text.split("</div>")[0] for text in predictions_new]
predictions_new

['Во понеделник, научници од медицинскиот факултет на универзитетот Стенфорд објавиле изумот на ново дијагностичко орудие што може да подредува клетки по тип: малиот штампан чип што може да се произведува со стандардни штампачи за туш за можно една американска цента.',
 'Лидните истражувачи велат дека ова може да донесе рано откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со ниско приход, каде што оцетот за болести како што е ракот на дојката може да биде половина од оној во земјите со поголем приход.',
 'JAS 39C Gripen се срушил на пистата околу 9:30 утрински часови по локално време (0230 UTC) и експлодираше, затворајќи аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како еден од пилотите на воздухопловната база во Тајланд.',
 'Локалните медиуми извештавале за пожарна кола која се преокренала додека одговарала на повик.',
 '28-годишниот Видал се приклучил на Барса три сезони преди, од Севиља.',
 'Од кога се преселил во каталонската престолнина,

In [103]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [104]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.18589903513446365,
 'precisions': [0.5240032546786005,
  0.24851569126378287,
  0.13684676705048715,
  0.07645968489341984],
 'brevity_penalty': 0.9675834342650409,
 'length_ratio': 0.9680976762504924,
 'translation_length': 2458,
 'reference_length': 2539}

In [105]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 49.10
